# 01 — Datos, procedencia y partición temporal

La unidad de regresión es el diseño definido por sus siete entradas exactas. Las apariciones poblacionales describen una sola trayectoria. Este notebook lee la preparación y el informe del estudio seleccionado; no prepara datos ni ajusta modelos. El enfoque científico y sus límites están en [CONTEXT.md](../CONTEXT.md) y la ficha del dataset en [data/README.md](../data/README.md).

In [ ]:
%matplotlib inline
%config InlineBackend.figure_formats = {"png", "svg"}
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from truss_research.evaluacion.presentacion import notebook_context

context = notebook_context()
CONFIG = context.config
STUDY_DIR, REPORT_DIR = context.study_dir, context.report_dir
DATA_DIR = CONFIG.data_dir
display(
    Markdown(
        f"**Estudio:** `{STUDY_DIR.name}` — **etiqueta:** `{CONFIG.label}` — **informe:** `{REPORT_DIR.name}`"
    )
)
if CONFIG.label == "comprobacion":
    display(
        Markdown(
            "**Comprobación técnica: sus métricas no constituyen resultados de tesis.**"
        )
    )

## 1. Entradas y respuestas

`web_config` es nominal. Los identificadores y las respuestas no son entradas del predictor. Masa se expresa en kg, desplazamiento Z en m y utilización promedio como ratio. El significado físico procede de las declaraciones registradas; no se deduce solo del Excel.

In [ ]:
from truss_research.cerchas.schemas import INPUT_COLUMNS, OUTPUT_COLUMNS
from truss_research.datos.procesados import read_manifest

manifest = read_manifest(DATA_DIR)
training = pd.read_csv(DATA_DIR / "train.csv", float_precision="round_trip")
full = pd.read_csv(DATA_DIR / "full_clean.csv", float_precision="round_trip")
display(
    pd.DataFrame(
        {
            "variable": [*INPUT_COLUMNS, *OUTPUT_COLUMNS],
            "rol": ["entrada"] * len(INPUT_COLUMNS)
            + ["respuesta"] * len(OUTPUT_COLUMNS),
        }
    )
)
display(training[["iter_id", *INPUT_COLUMNS, *OUTPUT_COLUMNS]].head())

## 2. Apariciones, identidad y duplicados

La identidad usa igualdad exacta de las siete entradas, sin redondear alturas. La preparación comprueba respuestas coincidentes antes de conservar el representante de menor generación y solución. La frecuencia no pondera el entrenamiento.

Cada generación registra una población; el índice de solución no establece orden de evaluación. Cruce, mutación y elitismo requieren evidencia del algoritmo original. Una reaparición no prueba otra llamada FEM.

In [ ]:
display(pd.Series(manifest["design_audit"], name="Auditoría del origen"))
membership = pd.read_csv(DATA_DIR / "design_membership.csv")
display(membership[["iter_id", "representative_iter_id", "split"]].head(8))

## 3. Primera aparición y cobertura temporal

El protocolo principal asigna cada diseño por su primera generación a ajuste, calibración, validación o prueba. Un superviviente conserva su rol original. Las categorías no se trasladan para equilibrar bloques. La sensibilidad de interpolación por diseños únicos es un experimento separado.

Los tamaños y categorías siguientes corresponden al manifiesto de este estudio. Las categorías ausentes en prueba quedan fuera de su evaluación final; las semillas no compensan esa falta de cobertura.

In [ ]:
diagnostics = manifest["diagnostics"]
partitions = pd.DataFrame(
    [
        {
            "conjunto": name,
            "diseños": row["rows"],
            "primera_generación_mínima": min(row["generations"]),
            "primera_generación_máxima": max(row["generations"]),
        }
        for name, row in diagnostics.items()
    ]
)
display(partitions)
display(pd.Series(manifest["design_overlap"], name="Diseños compartidos"))
coverage = pd.read_csv(REPORT_DIR / "category_coverage.csv")
display(coverage)
counts = pd.DataFrame(
    {
        name: row["web_config_counts"]
        for name, row in diagnostics.items()
        if name != "full"
    }
).fillna(0)
counts.plot.bar(
    figsize=(9, 4), ylabel="Diseños", title="Cobertura por topología y bloque"
)
plt.tight_layout()
plt.show()

## 4. Umbrales y distribución de entrenamiento

L/700 es un criterio operativo adoptado, no una restricción demostrada del generador original. Se conservan los diseños que incumplen umbrales. La utilización promedio no certifica cada barra; la ausencia de ratios mayores que uno no demuestra un filtrado previo del optimizador.

Los conteos globales describen el archivo ya examinado. Las decisiones de ajuste se apoyan en entrenamiento; los histogramas siguientes usan únicamente ese bloque.

In [ ]:
limits = manifest["limits"]
z_limit = limits["displacement_limit_m"]
z_compliance = (
    full["z_disp_m"].le(z_limit)
    if z_limit is not None
    else pd.Series(True, index=full.index)
)
display(
    pd.Series(
        {
            "diseños_únicos": len(full),
            "límite_Z_m": z_limit,
            "cumplen_Z": int(z_compliance.sum()),
            "exceden_Z": int((~z_compliance).sum()),
            "utilización_máxima_observada": full["util_ratio"].max(),
            "exceden_utilización": int(
                full["util_ratio"].gt(limits["utilization_limit"]).sum()
            ),
        }
    )
)
training[list(OUTPUT_COLUMNS)].hist(figsize=(11, 3), bins=15)
plt.tight_layout()
plt.show()

## 5. Procedencia y alcance

La ficha conserva declaraciones humanas y sus pendientes. Faltan condiciones para reconstruir y certificar nuevas configuraciones. El documento histórico `docs/SCOPE.md` citado en el registro no está disponible; recuperar la evidencia original sigue pendiente. Los resultados describen una sola trayectoria previamente examinada.

In [ ]:
display(pd.read_csv(REPORT_DIR / "physical_provenance.csv"))
display(Markdown((REPORT_DIR / "EVIDENCE.md").read_text(encoding="utf-8")))